In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="02-visual-search",
    notebook_name="01_visual_search_system_design.ipynb"
)

# Visual Search System Design

## What We Are Building

Imagine you take a photo of a cool pair of sneakers you see on the street. You want to find where to buy them -- or find sneakers that look just like them. **Visual search** lets you search using a picture instead of typing words.

In this notebook, we design a complete visual search system (like Pinterest's visual search) from scratch. We cover:

1. Problem definition and clarification
2. Framing it as an ML task
3. Data pipeline
4. Feature engineering
5. Model architecture
6. Training approach
7. Evaluation metrics
8. Serving architecture

---

## 1. Problem Definition & Clarification

### The Problem Statement

Design a visual search system that:
- Takes a query image (or image crop) from a user
- Retrieves images visually similar to the query
- Ranks results from most similar to least similar
- Works at massive scale (100-200 billion images)
- Returns results quickly (sub-second latency)

### Key Clarifications (Always Ask These in an Interview!)

| Question | Answer | Why It Matters |
|----------|--------|----------------|
| Ranked results? | Yes, most similar first | This makes it a ranking problem |
| Images only? | Yes, no video/text queries | Simplifies the input modality |
| Support image crops? | Yes | Need to handle partial images |
| Personalized? | No | Same query = same results for all users |
| Use metadata (tags)? | No, pixels only | Pure visual similarity |
| Available user actions? | Impressions + clicks | Defines our interaction signal |
| Content moderation? | Out of scope | Can discuss as follow-up |
| Training data source? | User interactions (online) | Self-supervised + click data |
| Scale? | 100-200B images | Must use approximate search |

## 2. Frame as an ML Task

### ML Objective

**Accurately retrieve images that are visually similar to the query image.**

### Input/Output

```
Input:  Query image (or crop of an image)
Output: Ranked list of visually similar images
```

### ML Category: Ranking via Representation Learning

Think of it this way: imagine every image is a point on a giant map. Two photos of golden retrievers would be close together on this map, while a photo of a pizza would be far away. The "map" is called an **embedding space**, and each point is an **embedding vector**.

```
Representation Learning:
    Image --> [Neural Network] --> Embedding Vector (e.g., 256 numbers)
    
    Similar images    --> nearby points in embedding space
    Dissimilar images --> far apart points in embedding space
```

To rank images:
1. Convert query image to embedding
2. Compute similarity between query embedding and all indexed embeddings
3. Return images sorted by similarity score

## 3. System Architecture Diagram

Let's visualize the complete system architecture.

In [ ]:
# Let's draw the system architecture
# We'll use matplotlib to create a clear diagram

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(2, 1, figsize=(16, 14))

# ============================================================
# TOP: Prediction Pipeline (real-time)
# ============================================================
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.set_title('Prediction Pipeline (Real-Time Query Processing)', fontsize=16, fontweight='bold', pad=15)
ax.axis('off')

# Boxes for prediction pipeline
boxes_pred = [
    (0.3, 1.5, 'Query\nImage', '#FFE0B2'),
    (2.2, 1.5, 'Image\nPreprocessing', '#B3E5FC'),
    (4.1, 1.5, 'Embedding\nGeneration\n(CNN/ViT)', '#C8E6C9'),
    (6.0, 1.5, 'Nearest\nNeighbor\nSearch (ANN)', '#F8BBD0'),
    (7.9, 1.5, 'Re-ranking\nService', '#D1C4E9'),
    (9.2, 1.5, 'Results', '#FFE0B2'),
]

for (x, y, text, color) in boxes_pred:
    box = FancyBboxPatch((x, y), 1.4, 1.2, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + 0.7, y + 0.6, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
arrow_xs = [(1.7, 2.2), (3.6, 4.1), (5.5, 6.0), (7.4, 7.9)]
for (x1, x2) in arrow_xs:
    ax.annotate('', xy=(x2, 2.1), xytext=(x1, 2.1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Arrow from re-ranking to results
ax.annotate('', xy=(9.2, 2.1), xytext=(9.3 - 0.7, 2.1),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Index table connection
idx_box = FancyBboxPatch((5.5, 0.0), 2.0, 0.9, boxstyle='round,pad=0.1',
                         facecolor='#FFF9C4', edgecolor='#333333', linewidth=2)
ax.add_patch(idx_box)
ax.text(6.5, 0.45, 'Embedding\nIndex Table', ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(6.7, 1.5), xytext=(6.7, 0.9),
            arrowprops=dict(arrowstyle='->', color='#666', lw=1.5, ls='--'))

# ============================================================
# BOTTOM: Indexing Pipeline (offline/batch)
# ============================================================
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 4)
ax2.set_title('Indexing Pipeline (Offline / Batch Processing)', fontsize=16, fontweight='bold', pad=15)
ax2.axis('off')

boxes_idx = [
    (0.3, 1.5, 'All Platform\nImages', '#FFE0B2'),
    (2.5, 1.5, 'Image\nPreprocessing', '#B3E5FC'),
    (4.7, 1.5, 'Embedding\nGeneration\n(CNN/ViT)', '#C8E6C9'),
    (7.2, 1.5, 'Embedding\nIndex Table\n(FAISS/ScaNN)', '#FFF9C4'),
]

for (x, y, text, color) in boxes_idx:
    box = FancyBboxPatch((x, y), 1.8, 1.2, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333333', linewidth=2)
    ax2.add_patch(box)
    ax2.text(x + 0.9, y + 0.6, text, ha='center', va='center', fontsize=9, fontweight='bold')

arrow_xs2 = [(2.1, 2.5), (4.3, 4.7), (6.5, 7.2)]
for (x1, x2) in arrow_xs2:
    ax2.annotate('', xy=(x2, 2.1), xytext=(x1, 2.1),
                 arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# New image upload flow
new_img_box = FancyBboxPatch((0.3, 0.0), 1.8, 0.9, boxstyle='round,pad=0.1',
                             facecolor='#FFCCBC', edgecolor='#333333', linewidth=2)
ax2.add_patch(new_img_box)
ax2.text(1.2, 0.45, 'New Image\nUploaded', ha='center', va='center', fontsize=9, fontweight='bold')
ax2.annotate('', xy=(1.2, 1.5), xytext=(1.2, 0.9),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))

# Annotations
ax2.text(5.0, 3.5, 'When new images are uploaded, the indexing service computes their\n'
         'embeddings and adds them to the index to make them searchable.',
         ha='center', va='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round', facecolor='#E8EAF6', alpha=0.8))

plt.tight_layout()
plt.savefig('system_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Pipeline

### What Data Do We Have?

Our visual search system revolves around three types of data:

1. **Images** -- the actual image files plus metadata (owner, upload time, tags)
2. **Users** -- demographic info (age, gender, city, country)
3. **User-Image Interactions** -- impressions and clicks

Let's look at each one:

In [ ]:
import pandas as pd

# ---- Image Metadata ----
images_df = pd.DataFrame({
    'image_id': [1, 2, 3],
    'owner_id': [8, 5, 19],
    'upload_time': [1658451341, 1658451841, 1658821820],
    'manual_tags': ['Zebra', 'Pasta, Food, Kitchen', 'Children, Family, Party']
})

print("=== Image Metadata ===")
print(images_df.to_string(index=False))
print()

# ---- User Data ----
users_df = pd.DataFrame({
    'user_id': [1, 2, 3],
    'username': ['johnduo', 'hs2008', 'alexish'],
    'age': [26, 49, 16],
    'gender': ['M', 'M', 'F'],
    'city': ['San Jose', 'Paris', 'Rio'],
    'country': ['USA', 'France', 'Brazil']
})

print("=== User Data ===")
print(users_df.to_string(index=False))
print()

# ---- Interaction Data ----
interactions_df = pd.DataFrame({
    'user_id': [8, 6, 91],
    'query_image_id': [2, 3, 5],
    'displayed_image_id': [6, 9, 1],
    'position': [1, 2, 2],
    'interaction_type': ['Click', 'Click', 'Impression']
})

print("=== User-Image Interactions ===")
print(interactions_df.to_string(index=False))
print()
print("Key insight: Clicks tell us 'this user thought these images were similar.'")
print("Impressions tell us 'this image was shown but NOT clicked.'")

## 5. Feature Engineering (Image Preprocessing)

Before we can feed images into our model, we need to preprocess them. Think of it like preparing ingredients before cooking -- you need to wash, chop, and measure everything first.

### The Four Preprocessing Steps

1. **Resize** to fixed dimensions (e.g., 224x224) -- models need consistent input sizes
2. **Scale** pixel values to [0, 1] -- raw pixels are 0-255, which is too large
3. **Z-score normalize** -- center around mean=0, std=1 for stable training
4. **Ensure consistent color mode** -- always RGB, not CMYK or grayscale

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# ======================================================================
# Standard image preprocessing pipeline for visual search
# This is what both the indexing and prediction pipelines use.
# ======================================================================

# The transform pipeline:
# 1. Resize to 224x224 (standard for ResNet/ViT)
# 2. Convert to tensor (scales pixels from [0,255] to [0,1] automatically)
# 3. Normalize with ImageNet mean and std (Z-score normalization)

preprocess = transforms.Compose([
    transforms.Resize(256),               # Resize shorter side to 256
    transforms.CenterCrop(224),            # Crop center 224x224
    transforms.ToTensor(),                 # [0,255] -> [0,1] + HWC -> CHW
    transforms.Normalize(                  # Z-score with ImageNet stats
        mean=[0.485, 0.456, 0.406],        # ImageNet channel means
        std=[0.229, 0.224, 0.225]           # ImageNet channel stds
    ),
])

# Let's demonstrate with a synthetic image
# (In practice, you'd load a real image from disk)
dummy_image = Image.fromarray(
    np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8)
)
dummy_image = dummy_image.convert('RGB')  # Step 4: Ensure RGB

# Apply preprocessing
preprocessed = preprocess(dummy_image)

print(f"Original image shape: {np.array(dummy_image).shape}")  # (300, 400, 3)
print(f"Preprocessed tensor shape: {preprocessed.shape}")       # torch.Size([3, 224, 224])
print(f"Pixel value range: [{preprocessed.min():.3f}, {preprocessed.max():.3f}]")
print(f"Mean per channel: {preprocessed.mean(dim=(1,2))}")
print(f"Std per channel: {preprocessed.std(dim=(1,2))}")
print()
print("After preprocessing:")
print("  - Fixed size: 224x224 (model requirement)")
print("  - CHW format: channels first (PyTorch convention)")
print("  - Normalized: approximately mean=0, std=1 per channel")

In [ ]:
# ======================================================================
# Data Augmentation for Self-Supervised Training (SimCLR-style)
# ======================================================================
# When we use self-supervision, we create "positive pairs" by augmenting
# the same image in two different ways. The model learns that these
# two augmented versions should have similar embeddings.

# Think of it like this: if you take a photo of a dog and then 
# rotate it slightly, crop it, and change the brightness -- it's 
# still a photo of the same dog! The model should recognize that.

simclr_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),  # Random crop + resize
    transforms.RandomHorizontalFlip(p=0.5),               # 50% chance flip
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)       # Color distortion
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),                    # 20% chance grayscale
    transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),  # Blur
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Create two augmented views of the same image
view1 = simclr_augmentation(dummy_image)
view2 = simclr_augmentation(dummy_image)

print("SimCLR-style Self-Supervised Training:")
print(f"  View 1 shape: {view1.shape}")
print(f"  View 2 shape: {view2.shape}")
print(f"  These are two different augmentations of the SAME image.")
print(f"  The model should learn that they have similar embeddings.")
print()
print("Augmentation operations used:")
print("  1. Random crop + resize (different regions of the image)")
print("  2. Random horizontal flip")
print("  3. Color jitter (brightness, contrast, saturation, hue)")
print("  4. Random grayscale conversion")
print("  5. Gaussian blur")

## 6. Model Architecture

### The Embedding Model

Our model takes an image and produces an embedding vector. Think of it as converting a picture into a list of numbers that captures "what the picture looks like."

```
Input Image (224x224x3)
    |
    v
[Convolutional Layers]  <-- Extract visual features (edges, textures, shapes, objects)
    |
    v
[Global Average Pooling] <-- Compress spatial dimensions
    |
    v
[Fully Connected Layer]  <-- Map to embedding dimension
    |
    v
[L2 Normalization]       <-- Normalize to unit sphere
    |
    v
Embedding Vector (e.g., 256-D)
```

### Architecture Choices

| Architecture | Description | When to Use |
|-------------|-------------|-------------|
| **ResNet-50** | CNN with residual connections | Good default, efficient, battle-tested |
| **ResNet-152** | Deeper ResNet | More capacity, better accuracy, slower |
| **ViT (Vision Transformer)** | Transformer applied to image patches | State-of-the-art, needs more data |
| **EfficientNet** | Optimized CNN architecture | Best accuracy/compute trade-off |

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ======================================================================
# Visual Search Embedding Model
# 
# Architecture: ResNet-50 backbone + projection head
# Output: 256-dimensional normalized embedding vector
# ======================================================================

class VisualSearchModel(nn.Module):
    """
    Embedding model for visual search.
    
    Takes an image and produces a normalized embedding vector.
    Similar images will have embeddings close together (high cosine similarity).
    
    Think of it like this:
    - The ResNet backbone is like a person who looks at the image and 
      notices all the important features (shapes, colors, textures, objects).
    - The projection head is like summarizing all those observations 
      into a compact "fingerprint" of the image.
    - L2 normalization puts all fingerprints on a unit sphere so we can 
      compare them fairly using cosine similarity.
    """
    
    def __init__(self, embedding_dim=256, pretrained=True):
        super().__init__()
        
        # Backbone: ResNet-50 pretrained on ImageNet
        # We remove the final classification layer
        resnet = models.resnet50(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC layer
        # Output of backbone: (batch_size, 2048, 1, 1)
        
        # Projection head: maps 2048-D features to embedding_dim-D
        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, embedding_dim),
        )
    
    def forward(self, x):
        # x: (batch_size, 3, 224, 224)
        
        # Extract features with backbone
        features = self.backbone(x)          # (batch_size, 2048, 1, 1)
        features = features.flatten(1)        # (batch_size, 2048)
        
        # Project to embedding space
        embedding = self.projection(features)  # (batch_size, embedding_dim)
        
        # L2 normalize so all embeddings lie on a unit sphere
        # This makes cosine similarity = dot product
        embedding = nn.functional.normalize(embedding, p=2, dim=1)
        
        return embedding


# Create model and inspect
model = VisualSearchModel(embedding_dim=256, pretrained=False)  # pretrained=False to avoid download

# Test with a dummy batch
dummy_batch = torch.randn(4, 3, 224, 224)  # 4 images
embeddings = model(dummy_batch)

print(f"Input shape:     {dummy_batch.shape}")    # [4, 3, 224, 224]
print(f"Embedding shape: {embeddings.shape}")     # [4, 256]
print(f"Embedding norm:  {embeddings.norm(dim=1)}")  # Should be [1, 1, 1, 1]
print()
print("Each image is now represented as a 256-dimensional unit vector.")
print("We can compare any two images using cosine similarity (dot product).")

In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.backbone.parameters())
projection_params = sum(p.numel() for p in model.projection.parameters())

print(f"Model Parameter Count:")
print(f"  Backbone (ResNet-50):  {backbone_params:>12,} parameters")
print(f"  Projection Head:       {projection_params:>12,} parameters")
print(f"  Total:                 {total_params:>12,} parameters")
print()
print("Why ResNet-50?")
print("  - Battle-tested: used in production at Pinterest, Google, etc.")
print("  - Good accuracy/speed tradeoff")
print("  - Easy to fine-tune from ImageNet pretrained weights")
print("  - For even better results, consider ViT-B/16 or EfficientNet-B7")

## 7. Model Training: Contrastive Learning

### The Core Idea

Imagine a quiz game:
- "Here's a photo of a golden retriever (query). Which of these 10 images is most similar?"
- One of the 10 is another golden retriever photo (positive).
- The other 9 are photos of cats, cars, buildings, etc. (negatives).

The model learns to pick the right answer. Over millions of such quizzes, it learns what makes images similar.

### Three Ways to Get Positive Pairs

| Method | Description | Pros | Cons |
|--------|------------|------|------|
| **Human Judgment** | Annotators manually label similar pairs | Most accurate | Expensive, slow |
| **User Clicks** | If user clicked image B after searching with image A, they are similar | Automatic | Noisy, sparse |
| **Self-Supervision** | Augment query image (rotate, crop, etc.) to create positive | Free, scalable, not noisy | Augmented != real similar |

**Our choice: Self-supervision** (SimCLR/MoCo style) because we have billions of images and it is free.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================================================================
# Contrastive Loss (NT-Xent / InfoNCE)
# 
# This is the loss function used in SimCLR-style contrastive training.
# It has three steps:
#   1. Compute similarity scores between query and all candidates
#   2. Apply softmax to get probabilities
#   3. Compute cross-entropy loss against the ground truth
# ======================================================================

class ContrastiveLoss(nn.Module):
    """
    NT-Xent (Normalized Temperature-scaled Cross Entropy) Loss.
    
    Think of it like this:
    - We have a query image and N candidate images (1 positive + N-1 negatives).
    - We compute how similar the query is to each candidate.
    - We want the positive to have the highest similarity score.
    - The temperature parameter controls how "sharp" the distribution is:
      - Low temperature (0.05): very sharp, forces model to be very confident
      - High temperature (1.0): softer, more tolerant of ambiguity
    """
    
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, query_embeddings, candidate_embeddings, positive_indices):
        """
        Args:
            query_embeddings: (batch_size, embedding_dim) - embeddings of query images
            candidate_embeddings: (batch_size, n_candidates, embedding_dim) - embeddings of candidates
            positive_indices: (batch_size,) - index of the positive image for each query
        
        Returns:
            loss: scalar - the contrastive loss
        """
        batch_size = query_embeddings.shape[0]
        
        # Step 1: Compute similarities (dot product since embeddings are L2-normalized)
        # query_embeddings: (B, D) -> (B, 1, D)
        # candidate_embeddings: (B, N, D)
        # similarities: (B, N)
        similarities = torch.bmm(
            query_embeddings.unsqueeze(1),          # (B, 1, D)
            candidate_embeddings.transpose(1, 2)    # (B, D, N)
        ).squeeze(1)                                # (B, N)
        
        # Scale by temperature
        similarities = similarities / self.temperature
        
        # Steps 2 & 3: Softmax + Cross-entropy (combined in F.cross_entropy)
        loss = F.cross_entropy(similarities, positive_indices)
        
        return loss


# ---- Demo ----
torch.manual_seed(42)
loss_fn = ContrastiveLoss(temperature=0.07)

# Simulate a batch:
# - 4 query images
# - 10 candidates each (1 positive + 9 negatives)
# - 256-dimensional embeddings
batch_size, n_candidates, emb_dim = 4, 10, 256

query_emb = F.normalize(torch.randn(batch_size, emb_dim), dim=1)
cand_emb = F.normalize(torch.randn(batch_size, n_candidates, emb_dim), dim=2)
positive_idx = torch.tensor([2, 5, 0, 7])  # Index of positive for each query

loss = loss_fn(query_emb, cand_emb, positive_idx)
print(f"Contrastive Loss: {loss.item():.4f}")
print(f"  (Random guess would give loss = -ln(1/10) = {-torch.log(torch.tensor(1/10)).item():.4f})")
print()
print("How the loss works:")
print("  1. Compute cosine similarity between query and each candidate")
print("  2. Divide by temperature (0.07) to sharpen the distribution")
print("  3. Apply softmax to get probabilities")
print("  4. Cross-entropy: penalize when the positive image has low probability")
print()
print("During training, the model learns to:")
print("  - Push positive pairs CLOSER in embedding space")
print("  - Push negative pairs FURTHER APART in embedding space")

In [ ]:
# ======================================================================
# Simplified Training Loop
# ======================================================================

def train_visual_search_model(model, dataloader, num_epochs=10, lr=3e-4, temperature=0.07):
    """
    Train the visual search embedding model with contrastive learning.
    
    In practice (at Pinterest scale), this would:
    - Run on multiple GPUs (distributed training)
    - Use a larger batch size (4096+) for more negatives
    - Train for many more epochs
    - Use learning rate scheduling (warmup + cosine decay)
    - Use mixed precision (fp16) for efficiency
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = ContrastiveLoss(temperature=temperature)
    
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        num_batches = 0
        
        for batch in dataloader:
            query_images = batch['query']           # (B, 3, 224, 224)
            candidate_images = batch['candidates']   # (B, N, 3, 224, 224)
            positive_indices = batch['positive_idx'] # (B,)
            
            # Get query embeddings
            query_emb = model(query_images)  # (B, embedding_dim)
            
            # Get candidate embeddings
            B, N, C, H, W = candidate_images.shape
            candidate_flat = candidate_images.view(B * N, C, H, W)
            cand_emb = model(candidate_flat)  # (B*N, embedding_dim)
            cand_emb = cand_emb.view(B, N, -1)  # (B, N, embedding_dim)
            
            # Compute loss
            loss = loss_fn(query_emb, cand_emb, positive_indices)
            
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / max(num_batches, 1)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("Training loop defined.")
print()
print("Key hyperparameters to tune:")
print("  - embedding_dim: 128, 256, or 512 (larger = more expressive, more memory)")
print("  - temperature: 0.05 to 0.1 (controls sharpness of similarity distribution)")
print("  - learning rate: 1e-4 to 3e-4 (with warmup schedule)")
print("  - batch size: larger is better (more negatives per query)")
print("  - number of negatives: more negatives = harder task = better embeddings")

## 8. Evaluation Metrics

### Offline Metrics

We have an evaluation dataset where each query image has candidate images with ground truth relevance scores (0-5).

Let's implement and understand the key metrics:

In [ ]:
import numpy as np

# ======================================================================
# Evaluation Metrics for Ranking Systems
# ======================================================================

def mrr(ranked_relevances_list):
    """
    Mean Reciprocal Rank.
    
    Looks at the position of the FIRST relevant item in each ranked list.
    
    Limitation: Only considers the first relevant item.
    If list A has 5 relevant items and list B has 1, they could get the same MRR.
    """
    rr_sum = 0
    for relevances in ranked_relevances_list:
        for i, rel in enumerate(relevances):
            if rel > 0:  # First relevant item
                rr_sum += 1.0 / (i + 1)
                break
    return rr_sum / len(ranked_relevances_list)


def precision_at_k(relevances, k):
    """
    Precision@k: what fraction of the top-k items are relevant?
    
    Limitation: Does not consider ranking order within the top-k.
    [relevant, irrelevant, relevant] has the same precision@3 as
    [irrelevant, relevant, relevant].
    """
    top_k = relevances[:k]
    return sum(1 for r in top_k if r > 0) / k


def recall_at_k(relevances, k, total_relevant):
    """
    Recall@k: what fraction of ALL relevant items are in the top-k?
    
    Limitation: Denominator can be huge (millions of similar images).
    """
    top_k = relevances[:k]
    return sum(1 for r in top_k if r > 0) / total_relevant


def average_precision(relevances):
    """
    Average Precision (AP): averages precision@k for each relevant item.
    
    Better than precision@k because it considers ranking order.
    Limitation: Designed for binary relevance (relevant/irrelevant).
    """
    num_relevant = 0
    precision_sum = 0
    
    for i, rel in enumerate(relevances):
        if rel > 0:
            num_relevant += 1
            precision_sum += num_relevant / (i + 1)
    
    if num_relevant == 0:
        return 0.0
    return precision_sum / num_relevant


def ndcg(relevance_scores, k=None):
    """
    Normalized Discounted Cumulative Gain (nDCG).
    
    THE metric we use for visual search because:
    - Works with continuous relevance scores (not just binary)
    - Measures ranking quality (penalizes relevant items ranked low)
    - Normalized to [0, 1] by comparing against ideal ranking
    
    Formula:
        DCG  = sum(rel_i / log2(i + 1)) for i = 1..k
        IDCG = DCG of the ideal ranking (sorted by relevance)
        nDCG = DCG / IDCG
    """
    if k is not None:
        relevance_scores = relevance_scores[:k]
    
    # Compute DCG
    dcg = sum(
        rel / np.log2(i + 2)  # i+2 because log2(1) = 0
        for i, rel in enumerate(relevance_scores)
    )
    
    # Compute IDCG (ideal ranking = sorted descending)
    ideal_scores = sorted(relevance_scores, reverse=True)
    idcg = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(ideal_scores)
    )
    
    if idcg == 0:
        return 0.0
    return dcg / idcg


# ======================================================================
# Example: Compare all metrics
# ======================================================================

# Simulated search results with ground truth relevance scores (0-5)
# Higher = more relevant to the query image
model_ranking = [0, 5, 1, 4, 2]   # Model's ranking (not ideal)
ideal_ranking = [5, 4, 2, 1, 0]   # Perfect ranking

print("Model's ranked results with relevance scores:")
print(f"  Positions:  [1, 2, 3, 4, 5]")
print(f"  Relevances: {model_ranking}")
print(f"  (Ideal):    {ideal_ranking}")
print()

# Binary relevances (relevant if score > 0)
binary_rel = [1 if r > 0 else 0 for r in model_ranking]

print(f"MRR (for this single list):     {1.0 / (next(i+1 for i, r in enumerate(model_ranking) if r > 0)):.4f}")
print(f"Precision@3:                    {precision_at_k(binary_rel, 3):.4f}")
print(f"Recall@3 (4 total relevant):    {recall_at_k(binary_rel, 3, 4):.4f}")
print(f"Average Precision (binary):     {average_precision(binary_rel):.4f}")
print(f"nDCG (continuous scores):       {ndcg(model_ranking):.4f}")
print()
print("Why nDCG is best for visual search:")
print("  - Uses the ACTUAL relevance scores (0-5), not just relevant/irrelevant")
print("  - Penalizes putting a score-5 image at position 5 instead of position 1")
print("  - Normalized to [0,1] so we can compare across different queries")
print(f"  - Perfect nDCG = 1.0, our model gets {ndcg(model_ranking):.4f}")

In [ ]:
# ======================================================================
# Detailed nDCG Calculation (Step by Step, matching the PDF example)
# ======================================================================

print("=== Step-by-Step nDCG Calculation (from the PDF) ===")
print()

# Model's ranking and relevance scores
model_scores = [0, 5, 1, 4, 2]
print(f"Model's ranking:   Position 1  2  3  4  5")
print(f"Relevance scores:          {model_scores[0]}  {model_scores[1]}  {model_scores[2]}  {model_scores[3]}  {model_scores[4]}")
print()

# Step 1: Compute DCG
print("Step 1: Compute DCG")
print("  DCG = sum(rel_i / log2(i+1)) for each position")
dcg_terms = []
for i, rel in enumerate(model_scores):
    term = rel / np.log2(i + 2)
    dcg_terms.append(term)
    print(f"  Position {i+1}: {rel} / log2({i+2}) = {rel} / {np.log2(i+2):.4f} = {term:.4f}")
dcg = sum(dcg_terms)
print(f"  DCG = {dcg:.4f}")
print()

# Step 2: Compute IDCG (ideal ranking)
ideal_scores = sorted(model_scores, reverse=True)
print("Step 2: Compute IDCG (ideal ranking)")
print(f"  Ideal ranking: {ideal_scores}")
idcg_terms = []
for i, rel in enumerate(ideal_scores):
    term = rel / np.log2(i + 2)
    idcg_terms.append(term)
    print(f"  Position {i+1}: {rel} / log2({i+2}) = {rel} / {np.log2(i+2):.4f} = {term:.4f}")
idcg = sum(idcg_terms)
print(f"  IDCG = {idcg:.4f}")
print()

# Step 3: Divide
ndcg_val = dcg / idcg
print(f"Step 3: nDCG = DCG / IDCG = {dcg:.4f} / {idcg:.4f} = {ndcg_val:.4f}")
print()
print(f"Interpretation: The model's ranking is {ndcg_val:.1%} as good as the ideal ranking.")

### Online Metrics

Once the model is deployed, we track:

| Metric | Formula | What It Tells Us |
|--------|---------|------------------|
| **CTR** | Clicked images / Total suggested images | Are users finding relevant results? |
| **Time Spent** | Average daily/weekly/monthly time on suggestions | Are users engaged with results? |

These are measured via A/B testing: deploy the new model to a fraction of users and compare metrics against the baseline.

## 9. Key Takeaways

### System Design Summary

```
1. CLARIFY: Images only, ranked, no personalization, 200B images
2. FRAME: Ranking problem -> Representation learning -> Embeddings
3. DATA: Images + Users + Interactions (clicks/impressions)
4. FEATURES: Resize -> Scale -> Normalize -> RGB
5. MODEL: CNN (ResNet-50) or ViT -> Projection head -> L2 norm -> 256-D embedding
6. TRAINING: Contrastive learning (SimCLR), self-supervised positive pairs
7. LOSS: Cosine similarity -> Softmax -> Cross-entropy
8. METRICS: Offline = nDCG, Online = CTR + Time Spent
9. SERVING:
   - Indexing pipeline: precompute embeddings for all images
   - Prediction pipeline: embed query -> ANN search -> re-rank
10. SCALE: ANN (FAISS/ScaNN) for sub-linear search over billions of images
```

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)